In [ ]:
#| export machine_learning.note_linking
from typing import Literal, Optional

from pathlib import Path
from os import PathLike

from trouver.obsidian.footnotes import identify_available_footnote_numbers
from trouver.obsidian.links import links_from_text, LinkType, ObsidianLink
from trouver.machine_learning.note_data import NoteLinkEnum, NoteData, note_data_order_cmp
from trouver.notation.parse import notation_in_note, main_of_notation
from trouver.personal_vault.note_type import (
    PersonalNoteTypeEnum, note_is_of_type
)
from trouver.helper import latex_str_is_likely_in_latex_str, latex_str_in_latex_str_fuzz_metric
from trouver.helper.regex import latex_indices
from trouver.obsidian.vault import VaultNote
from trouver.personal_vault.reference import index_note_for_reference
from trouver.personal_vault.notes import notes_linked_in_notes_linked_in_note
from trouver.obsidian.file import MarkdownFile
from trouver.notation.in_standard_info_note import notation_notes_linked_in_see_also_section


In [ ]:
from trouver.machine_learning.note_linking import MultiLabelPipeline, predict_note_linking, consolidate_note_linking_predictions_into_cache

In [ ]:
from unittest.mock import MagicMock
from unittest.mock import patch as mock_patch

## Identify notation notes that should be embedded as footnotes in information notes or linked in other notation notes

In [ ]:
#| export machine_learning.note_linking
def similar_notat_notes_in_note(
        origin_note: VaultNote, # Either an info or a notat note
        notation_notes: VaultNote | list[VaultNote], # The notation notes that are considered to be 
        threshold: float = 0.8, 
        ) -> list[VaultNote]: # The notation notes whose notations are determined to be similar to notations used in `origin_note`.
    """
    Determine which notation notes introduce notations which resemble notations used
    in `origin_note`.

    This is a fuzzy function purely based on the str value of the notation and the text of `origin_note` and does not use ML predictions. 
    """
    if isinstance(notation_notes, VaultNote):
        notation_notes = [notation_notes]

    text = origin_note.text()
    indices = latex_indices(text)
    latex_texts_in_origin_note: list[str] = []
    for start, end in indices:
        latex_text = text[start:end]
        latex_text = latex_text.strip('$ ')
        latex_texts_in_origin_note.append(latex_text)
    matching_notat_notes: list[VaultNote] = []
    for notation_note in notation_notes:
        if notation_note.name == origin_note.name:
            continue
        notation: str = notation_in_note(notation_note, include_dollar_signs=False)
        for latex_text in latex_texts_in_origin_note:
            if latex_str_is_likely_in_latex_str(notation, latex_text, threshold=threshold):
                matching_notat_notes.append(notation_note)
                break
    return matching_notat_notes
        

In [ ]:
#| TODO: test

In [ ]:
#| export machine_learning.note_linking
def locate_footnote_embedded_notation_link(
        origin_note: VaultNote, # An info note 
        notation_note: VaultNote, # The notation notes that are considered to be 
        locate_by: Literal['first', 'best'] = 'best', # If `'first'`, then the first latex string for which the `latex_str_in_latex_str_fuzz_metric` score exceed threshold is used as the location. If `'best'` or if no such latex string exists, then the latex string giving the greatest score is used as the location.
        threshold: float = 0.8,
        ) -> int | None: # The index in `origin_note.text()` at which the footnote to an embedded link to `notation_note` should be added. If the main note of `notation_note` is `origin_note`, then `None`.
    """
    Determine where in `origin_note` a footnote to an embedded link to `notation_note` should be added.

    Such a location would be at the end of the closing of a latex string in the text. 

    This is a fuzzy function purely based on the str value of the notation and the text of `origin_note` and does not use ML predictions. 
    """
    main_note = main_of_notation(notation_note, as_note=False)
    if main_note and origin_note.name == main_note:
        return None
    notation: str = notation_in_note(notation_note, include_dollar_signs=False)
    text = origin_note.text()
    indices = latex_indices(text)
    scores: dict[int, float] = {} # Keys are end indices and values are scores of how likely it seems that the latex str seems to use the notation.
    for start, end in indices:
        latex_text = text[start:end]
        latex_text = latex_text.strip('$ ')
        score: float = latex_str_in_latex_str_fuzz_metric(notation, latex_text)
        if score > threshold and locate_by == 'first':
            return end
        else:
            scores[end] = score
    max_key = max(scores, key=scores.get)
    return max_key

In [ ]:
mock_origin_note = MagicMock()
mock_notation_note = MagicMock()
mock_origin_note.text.return_value = r"""For each integer $m$ and each transitive $G \leq S_m$, there are constants $C(G), Q(G)$, and $e(G)$ such that, for all $q>Q(G)$ coprime to $\#G$ and all $X>0$, 

$$N_G(\mathbb{F}_q(t),X) \leq C(G) X^{a(G)} \log(X)^{e(G)}$$


"""

mock_notation_note.text.return_value = r"""---
detect_regex: 
latex_in_original: ["a(G)"]
tags: [_meta/notation_note_named]
---
$a(G)$ [[ellenberg_tran_westerland_fnfcqsamcff_1. Introduction_ellenberg_tran_westerland_fnfcqsamcff|denotes]] $[\min_{G \setminus \{1 \}} ind(g)]^{-1}$ where $G$ is a transitive subgroup of $S_m$ and 

![[ellenberg_tran_westerland_fnfcqsamcff_1. Introduction_ellenberg_tran_westerland_fnfcqsamcff#^38959b]]

for $g \in S_m$.

For instance, if $G = S_m$, the minimal index is $1$, realized by transpositions, and so $a(S_m) = 1$.
- [$ind(g)$](ellenberg_tran_westerland_fnfcqsamcff_notation_ind_g_index_of_element_of_S_m.md)"""

with (mock_patch('__main__.latex_str_in_latex_str_fuzz_metric')
        as mock_latex_str_in_latex_str_fuzz_metric,
        mock_patch('__main__.main_of_notation') as mock_main_of_notation,
        mock_patch('__main__.notation_in_note') as mock_notation_in_note,
        ):
    mock_latex_str_in_latex_str_fuzz_metric.side_effect = [0, 0, 0, 0, 0, 0, 0, 1]
    mock_main_note = MagicMock()
    mock_main_of_notation.return_value = mock_main_note
    mock_notation_in_note.return_value = '$a(G)$'
    output = locate_footnote_embedded_notation_link(
        mock_origin_note, mock_notation_note, locate_by='best', threshold=0.8)
    print(output)
    print(mock_origin_note.text.return_value[:output])

222
For each integer $m$ and each transitive $G \leq S_m$, there are constants $C(G), Q(G)$, and $e(G)$ such that, for all $q>Q(G)$ coprime to $\#G$ and all $X>0$, 

$$N_G(\mathbb{F}_q(t),X) \leq C(G) X^{a(G)} \log(X)^{e(G)}$$


In [ ]:
#| export machine_learning.note_linking
def _where_to_add_notation_links(
        origin_note: VaultNote,
        relied_notes: list[VaultNote],
        locate_by: Literal['first', 'best'] = 'best',
        threshold: float = 0.8,
        ) -> dict[int, list[VaultNote]]:
    """
    Helper function to `add_notation_note_embedded_footnotes_to_info_note`.
    """
    where_to_add: dict[int, list[VaultNote]] = {} # Keys are end indices of latex str in `origin_note` and values are lists of VaultNote objects representing notation notes for which the embedded links should be added.
    for relied_note in relied_notes:
        location: int | None = locate_footnote_embedded_notation_link(
            origin_note, relied_note, locate_by, threshold)
        if location is None:
            continue
        if not location in where_to_add:
            where_to_add[location] = []
        where_to_add[location].append(relied_note)
    return where_to_add

In [ ]:
#| export machine_learning.note_linking
def _add_notation_note_embedded_footnotes(
        text: str,
        where_to_add: dict[int, list[VaultNote]],
        ) -> str:
    """
    Helper function to `add_notation_note_embedded_footnotes_to_info_note`.
    """
    reverse_sorted_locations: list[int] = sorted(where_to_add.keys(), reverse=True)
    # iterate in reverse to make sure that the modifications made along the way do not change
    # the indices of the locations.
    for location in reverse_sorted_locations:
        notation_notes_to_link = where_to_add[location]
        available_footnote_numbers: list[int] = identify_available_footnote_numbers(
            text, count=len(notation_notes_to_link))
        footnote_text = ''.join([f'[^{num}]' for num in available_footnote_numbers])
        footnote_mentions = '\n'.join(
            [f'[^{num}]: ![[{notat_note.name}]]'
             for num, notat_note in zip(available_footnote_numbers, notation_notes_to_link)])
        new_line_index = text.find('\n', location+1)
        if new_line_index == -1:
            new_line_index = len(text)
        pieces = [text[0:location], text[location:new_line_index], text[new_line_index:]]

        if location > 1 and text[location-2] == '$': # latex str ends with '$$'
            # pieces.append(f'\n\n{footnote_text}\n\n{footnote_mentions}\n\n')
            pieces.insert(2, f'\n\n{footnote_text}\n\n{footnote_mentions}\n\n')
            text = ''.join(pieces)
            # start a new line to add the footnotes and then start another
            # to add the footnote mentions.
        else: # latex str ends with '$'
            pieces.insert(2, f'\n\n{footnote_mentions}\n\n')
            pieces.insert(1, footnote_text)
            text = ''.join(pieces)
            # new_line_index = 
    return text
    

In [ ]:
#| hide
text = r"""$$asdf$$"""
mock_notat_note = MagicMock()
mock_notat_note.name = 'notat_note_name'
where_to_add = {8: [mock_notat_note]}
print(_add_notation_note_embedded_footnotes(text, where_to_add))

text = r"""asdf asdf $asdf$ asdf asdf 

fjfjfjfj
"""
mock_notat_note = MagicMock()
mock_notat_note.name = 'notat_note_name'
where_to_add = {16: [mock_notat_note]}
output = _add_notation_note_embedded_footnotes(text, where_to_add)
print(output)


text = r"""---
cssclass: clean-embeds
aliases: []
tags: [_meta/literature_note, _reference/18785, _meta/concept, _meta/proof]
---
# Topic[^1]

Theorem 2.1. The map $\mathrm{q} \mapsto \mathrm{q} \cap A$ defines a bijection from the set of prime ideals of $S^{-1} A$ and the set of prime ideals of A that do not intersect $S .$ The inverse map is $\mathfrak{p} \mapsto \mathfrak{p} S^{-1} A$.

Proof. See [1, Cor.11.20] or [2, Prop. 3.11.iv].

# See Also

# Meta
## References
![[_reference_18785]]

## Citations and Footnotes
[^1]: Sutherland, Theorem 2.1, Page 11"""

mock_notat_note.name = '18785_notation_S_minus_1_A_localization_of_a_commutative_ring_with_respect_to_a_multiplicative_subset'
where_to_add = {254: [mock_notat_note]}
output = _add_notation_note_embedded_footnotes(text, where_to_add)
print(output)


text = r"""

For each integer $m$ and each transitive $G \leq S_m$, there are constants $C(G), Q(G)$, and $e(G)$ such that, for all $q>Q(G)$ coprime to $\#G$ and all $X>0$, 

$$N_G(\mathbb{F}_q(t),X) \leq C(G) X^{a(G)} \log(X)^{e(G)}$$

blah blah

"""
mock_notat_note = MagicMock()
mock_notat_note.name = 'notat_note_name'
where_to_add = {224: [mock_notat_note]}
output = _add_notation_note_embedded_footnotes(text, where_to_add)
print(output)

$$asdf$$

[^1]

[^1]: ![[notat_note_name]]


asdf asdf $asdf$[^1] asdf asdf 

[^1]: ![[notat_note_name]]



fjfjfjfj

---
cssclass: clean-embeds
aliases: []
tags: [_meta/literature_note, _reference/18785, _meta/concept, _meta/proof]
---
# Topic[^1]

Theorem 2.1. The map $\mathrm{q} \mapsto \mathrm{q} \cap A$ defines a bijection from the set of prime ideals of $S^{-1} A$[^2] and the set of prime ideals of A that do not intersect $S .$ The inverse map is $\mathfrak{p} \mapsto \mathfrak{p} S^{-1} A$.

[^2]: ![[18785_notation_S_minus_1_A_localization_of_a_commutative_ring_with_respect_to_a_multiplicative_subset]]



Proof. See [1, Cor.11.20] or [2, Prop. 3.11.iv].

# See Also

# Meta
## References
![[_reference_18785]]

## Citations and Footnotes
[^1]: Sutherland, Theorem 2.1, Page 11


For each integer $m$ and each transitive $G \leq S_m$, there are constants $C(G), Q(G)$, and $e(G)$ such that, for all $q>Q(G)$ coprime to $\#G$ and all $X>0$, 

$$N_G(\mathbb{F}_q(t),X) \leq C(G) X^{a(G)} 

In [ ]:
#| export machine_learning.note_linking
def add_notation_note_embedded_footnotes_to_info_note(
        origin_note: VaultNote, # An info note
        relied_notes: Optional[VaultNote | list[VaultNote]] = None, # notation notes to add embedded footnotes for.
        cache: Optional[dict[str, dict[str, list[NoteLinkEnum]]]] = None, # The cache from which to identify the notation notes to add embedded footnotes for.
        locate_by: Literal['first', 'best'] = 'best',
        threshold: float = 0.8,
        ):
    """
    Modify the contents of `origin_note` to add footnotes to embedded links to `relied_notes`

    One of `relied_notes` or `cache` must be passed.
    """
    if relied_notes is None and cache is None:
        raise ValueError("Expected `relied_note` or `cache` to be specified, but both were `None`.")
    if relied_notes is None:
        if origin_note.name not in cache:
            print(f'`origin_note.name` is not in `cache`. `origin_note` is {origin_note}.')
            return
        cache = remove_nonexistent_note_names_from_cache(cache, origin_note.vault)
        relied_notes: list[VaultNote] = []
        for relied_note_name, link_enums in cache[origin_note.name].items():
            relied_note = VaultNote(origin_note.vault, name=relied_note_name)
            if not note_is_of_type(relied_note, PersonalNoteTypeEnum.NOTATION_NOTE):
                continue
            if NoteLinkEnum.INFO_TO_NOTAT_VIA_EMBEDDING in link_enums:
                relied_notes.append(relied_note)
    if isinstance(relied_notes, VaultNote):
        relied_notes = [relied_notes]

    # Try to only add embedded links to notation notes that do not already exist in `origin_note`
    origin_note_text = origin_note.text()
    embedded_links_in_text: list[ObsidianLink] = links_from_text(
        origin_note_text, ObsidianLink(
            is_embedded=True, file_name=-1, anchor=-1, custom_text=-1, link_type=LinkType.WIKILINK))
    embedded_note_names_in_text: set[str] = set([link.file_name for link in embedded_links_in_text])
    new_relied_notes: list[VaultNote] = [
        relied_note for relied_note in relied_notes if relied_note.name not in embedded_note_names_in_text]

    where_to_add: dict[int, list[VaultNote]] = _where_to_add_notation_links(
        origin_note, new_relied_notes, locate_by, threshold)
    new_text = _add_notation_note_embedded_footnotes(origin_note.text(), where_to_add)
    origin_note.write(new_text)

In [ ]:
# origin_note = VaultNote(vault, name='18785_Theorem 2.1')
# notat_note = VaultNote(vault, name='18785_notation_S_minus_1_A_localization_of_a_commutative_ring_with_respect_to_a_multiplicative_subset')
# add_notation_note_embedded_footnotes_to_info_note(
#     origin_note, notat_note,
#     )

In [ ]:
# print(origin_note.text())

## Sieve note pairs to predict on

Since the number of pairs of notes grows quadratically in the number of notes, it takes too much time to make predictions one-by-one. It should be useful to prioritize certain pairs over others.


In [ ]:
#| export machine_learning.note_linking
def sieve_potential_relied_notes(
        vault: PathLike,
        reference: str,
        origin_note: VaultNote, # an info note
        note_data: dict[str, NoteData],
        # potential_relied_notes: list[VaultNote],
        appendix_notes: list[VaultNote], # notes whose index notes are appendix notes
        cache: dict[str, dict[str, list[NoteLinkEnum]]],
        notation_similarity_threshold: float = 0.8, # The threshold that the similarity metric of a notion must exceed for the name of a notation note to be included in the output.
        skip_already_made_predictions: bool = True,
        ) -> set[str]: # Names of potential relied notes that may be good to predict note linking from `origin_note` for.`
    if origin_note.name not in note_data:
        print(f'`origin_note` was not in `note_data`. Perhaps a `origin_note` has been renamed at some point and it may be necessary to reload `note_data`. `origin_note`: {origin_note}.')
        return set()
    index_note: VaultNote = index_note_for_reference(vault, reference, update_cache=True)
    info_notes: list[VaultNote] = notes_linked_in_notes_linked_in_note(index_note, as_dict=False)
    appendix_note_names: set[str] = set([appendix_note.name for appendix_note in appendix_notes])

    relied_note_names = set()

    # Add an info not if it 
    # 1. is in the appendix or precedes `origin_note`, is a definition/notation note
    # 2. is in the same section and precedes `origin_note` and is a context note.
    # TODO: Automatically add a def/notat note if it precedes `origin_note` in a section by a little.
    # Add a notation note if it 
    # 1. looks similar to a substr in a latex str in the origin_note.
    for info_note in info_notes + appendix_notes:
        if not info_note.exists():
            continue
        if info_note.name not in note_data:
            # If this happens, it may be the case that `info_note` has been
            # renamed, but this has not been reflected in `note_data`.
            continue
        # Ignore `info_note` if it was already predicted on or it precedes `origin_note` and is not an appendix note. 
        if (skip_already_made_predictions
                and origin_note.name in cache
                and info_note.name in cache[origin_note.name]):
            continue
        if (note_data_order_cmp(note_data[origin_note.name], note_data[info_note.name]) <= 0
                and info_note.name not in appendix_note_names):
            continue

        mf = MarkdownFile.from_vault_note(info_note)
        # ignore non-definition/notation notes.
        if not (mf.has_tag('_auto/_meta/definition') or mf.has_tag('_auto/_meta/notation') or mf.has_tag('_meta/definition') or mf.has_tag('_meta/notation')):
            continue
        # admit context notes in the same section as `origin_note` that also precede `origin_note`.
        elif (mf.has_tag("_auto/_meta/context") or mf.has_tag('_meta/context')
                and note_data_order_cmp(note_data[origin_note.name], note_data[info_note.name]) >= 0
                and note_data[origin_note.name].section_num == note_data[info_note.name].section_num):
            relied_note_names.add(info_note.name)
            continue
        relied_note_names.add(info_note.name)
        # For each info note with notations, try to see if the notations resemble notations used in `origin_note`.
        notat_notes: list[VaultNote] = notation_notes_linked_in_see_also_section(
            info_note, vault, as_vault_notes=True)
        if skip_already_made_predictions:
            notat_notes = [
                notat_note for notat_note in notat_notes
                if not (origin_note.name in cache and notat_note.name in cache[origin_note.name])]
        notat_note_candidates: list[VaultNote] = []
        for notat_note in notat_notes:
            individual_notat_note: list[VaultNote] = similar_notat_notes_in_note(
                origin_note, notat_note, threshold=notation_similarity_threshold)
            if not individual_notat_note:
                continue
            relied_note_names.add(info_note.name)
            relied_note_names.add(notat_note.name)

        # For each info note with definitions, try to see if the definitions resemble phrases used in `origin_note`. 


    
    # # 2. find all context notes in the same section as origin_note
    # origin_index_note = index_note_of_note(origin_note)
    # section_notes: dict[str, VaultNote] = notes_linked_in_note(
    #     origin_index_note, as_dict=True)
    # relied_note_names.update(section_notes.keys())

    return relied_note_names


In [ ]:
#| hide
from unittest.mock import MagicMock, patch
from fastcore.test import *

def test_sieve_potential_relied_notes():
    # 1. Setup Mock Data
    vault = "/mock/vault"
    reference = "Ref1"
    
    origin_note = MagicMock(spec=VaultNote)
    origin_note.name = "OriginNote"
    origin_note.vault = vault
    
    info_note_1 = MagicMock(spec=VaultNote)
    info_note_1.name = "Info1"
    info_note_1.exists.return_value = True
    
    appendix_note = MagicMock(spec=VaultNote)
    appendix_note.name = "App1"
    appendix_note.exists.return_value = True
    
    # Simple data mocks
    mock_origin_data = MagicMock()
    mock_origin_data.section_num = 1
    mock_info1_data = MagicMock()
    mock_info1_data.section_num = 1
    
    note_data = {
        "OriginNote": mock_origin_data,
        "Info1": mock_info1_data,
        "App1": MagicMock()
    }

    def create_mock_mf(tags):
        mf = MagicMock()
        mf.has_tag.side_effect = lambda t: t in tags
        return mf

    # 2. Comprehensive Patching
    # We patch __main__ because that is where the function usually lives in a notebook
    with patch('__main__.index_note_for_reference') as mock_idx, \
         patch('__main__.notes_linked_in_notes_linked_in_note') as mock_links, \
         patch('__main__.MarkdownFile.from_vault_note') as mock_mf_from, \
         patch('__main__.note_data_order_cmp') as mock_cmp, \
         patch('__main__.notation_notes_linked_in_see_also_section', return_value=[]), \
         patch('__main__.similar_notat_notes_in_note', return_value=[]):

        # Configure the Index Note Mock
        idx_note_mock = MagicMock(spec=VaultNote)
        idx_note_mock.name = "_index_Ref1"
        idx_note_mock.exists.return_value = True # This stops the NoteDoesNotExistError
        mock_idx.return_value = idx_note_mock
        
        # Return our list of mocked VaultNotes
        mock_links.return_value = [info_note_1]
        
        # Test Case 1: Filter out notes that appear AFTER origin_note
        mock_cmp.return_value = -1 
        cache = {}
        res = sieve_potential_relied_notes(vault, reference, origin_note, note_data, [], cache)
        test_eq(len(res), 0)

        # Test Case 2: Include definition notes that appear BEFORE origin_note
        mock_cmp.return_value = 1 
        mock_mf_from.return_value = create_mock_mf(['_meta/definition'])
        res = sieve_potential_relied_notes(vault, reference, origin_note, note_data, [], cache)
        test_eq(res, {"Info1"})

        # Test Case 3: Appendix notes are included regardless of order
        mock_cmp.return_value = -1 
        mock_links.return_value = [] # Standard links empty
        mock_mf_from.return_value = create_mock_mf(['_meta/definition'])
        res = sieve_potential_relied_notes(vault, reference, origin_note, note_data, [appendix_note], cache)
        test_eq(res, {"App1"})

        # Test Case 4: Skip already made predictions
        cache = {"OriginNote": {"Info1": ["SOME_LINK"]}}
        mock_links.return_value = [info_note_1]
        mock_cmp.return_value = 1
        res = sieve_potential_relied_notes(vault, reference, origin_note, note_data, [], cache)
        test_eq(len(res), 0)

test_sieve_potential_relied_notes()

In [ ]:
#| export machine_learning.note_linking
def _predict_one_direction_and_consolidate_cache(
        origin_note: VaultNote,
        relied_note: VaultNote,
        cache: dict[str, dict[str, list[NoteLinkEnum]]],
        predictor: MultiLabelPipeline, 
        format: Literal['bert', 't5'],
        note_data: dict[str, NoteData] | None,
        batch_size: int, 
        skip_already_made_predictions: bool,
        threshold: float | dict[str, float],
        ) -> None:
    """
    """
    if (skip_already_made_predictions
            and origin_note.name in cache and relied_note.name in cache[origin_note.name]):
        return
    outputs: dict[str, list[NoteLinkEnum]] = predict_note_linking(
        origin_note, relied_note, predictor, format, note_data)
    consolidate_note_linking_predictions_into_cache(origin_note, outputs, cache)

In [ ]:
#| export machine_learning.note_linking
def _predict_batch_and_consolidate_cache(
        origin_note: VaultNote,
        relied_notes: list[VaultNote], # Changed to list
        cache: dict[str, dict[str, list[NoteLinkEnum]]],
        predictor: MultiLabelPipeline, 
        format: Literal['bert', 't5'],
        note_data: dict[str, NoteData] | None,
        batch_size: int = 32, 
        skip_already_made_predictions: bool = True,
        threshold: float | dict[str, float] = 0.5,
        ) -> None:
    
    if not relied_notes:
        return

    # predict_note_linking handles the batching internally now
    outputs = predict_note_linking(
        origin_note=origin_note, 
        relied_notes=relied_notes, 
        predictor=predictor, 
        format=format, 
        note_data=note_data,
        batch_size=batch_size,
        skip_already_made_predictions=skip_already_made_predictions,
        threshold=threshold
    )
    
    # Consolidate results into the cache
    consolidate_note_linking_predictions_into_cache(origin_note, outputs, cache)

In [ ]:
#| hide
from unittest.mock import MagicMock, patch
from fastcore.test import *

def test_predict_batch_and_consolidate_cache():
    # 1. Setup Mock Objects
    origin_note = MagicMock(spec=VaultNote)
    origin_note.name = "Origin"
    
    relied_1 = MagicMock(spec=VaultNote)
    relied_1.name = "Relied1"
    
    relied_2 = MagicMock(spec=VaultNote)
    relied_2.name = "Relied2"
    
    relied_notes = [relied_1, relied_2]
    
    # Mock parameters
    predictor = MagicMock(spec=MultiLabelPipeline)
    cache = {}
    note_data = {"Origin": MagicMock(), "Relied1": MagicMock(), "Relied2": MagicMock()}
    
    # 2. Patch the dependencies in __main__ (or the module where they are defined)
    with patch('__main__.predict_note_linking') as mock_predict, \
         patch('__main__.consolidate_note_linking_predictions_into_cache') as mock_consolidate:
        
        # Define what predict_note_linking returns
        mock_outputs = {
            "Relied1": [NoteLinkEnum.INFO_TO_INFO_IN_CONTENT],
            "Relied2": [NoteLinkEnum.NO_LINK]
        }
        mock_predict.return_value = mock_outputs
        
        # --- Test Case 1: Empty relied_notes ---
        _predict_batch_and_consolidate_cache(
            origin_note, [], cache, predictor, 'bert', note_data
        )
        # Should return early without calling dependencies
        test_eq(mock_predict.call_count, 0)
        
        # --- Test Case 2: Successful Batch Execution ---
        _predict_batch_and_consolidate_cache(
            origin_note, 
            relied_notes, 
            cache, 
            predictor, 
            format='bert', 
            note_data=note_data,
            batch_size=16,
            threshold=0.6
        )
        
        # Verify predict_note_linking was called with the right arguments
        mock_predict.assert_called_once_with(
            origin_note=origin_note,
            relied_notes=relied_notes,
            predictor=predictor,
            format='bert',
            note_data=note_data,
            batch_size=16,
            skip_already_made_predictions=True,
            threshold=0.6
        )
        
        # Verify consolidation was called with the results from prediction
        mock_consolidate.assert_called_once_with(origin_note, mock_outputs, cache)

test_predict_batch_and_consolidate_cache()

In [ ]:
# #| export machine_learning.note_linking
# def predict_on_relied_note_and_related_notat_notes(
#         origin_note: VaultNote,
#         relied_note: VaultNote,
#         cache: dict[str, dict[str, list[NoteLinkEnum]]], # The current cache of predictions, see `parse_link_cache_note` for example; this is used to skip predictions that have already been made. Moreover, the cache is updated based on the predictions made. 
#         predictor: MultiLabelPipeline,
#         format: Literal['bert', 't5'] = 'bert',
#         note_data: Optional[dict[str, NoteData]] = None,
#         skip_already_made_predictions: bool = True,
#         predict_reverse_too: bool = False,
#         threshold: float | dict[str, float] = 0.5,
#         ) -> None:
#     """
#     Update `cache` by making predictions from `origin_note` to `relied_note` (and vice versa).
#     Moreover, 
#     """
#     # predict `origin_note` to `relied_note``

#     _predict_one_direction_and_consolidate_cache(
#         origin_note, relied_note, cache, predictor,
#         format, note_data, skip_already_made_predictions, threshold)
#     if predict_reverse_too:
#         _predict_one_direction_and_consolidate_cache(
#             relied_note, origin_note, cache, predictor, format, note_data,
#             skip_already_made_predictions, threshold)

#     # For each relied note that is 1. an info note, 2. got predicted to be a relied note via info_to_info_via_notat, and 3. has a notation, predict whether the relevant notation notes ought to be linked.
#     if not origin_note.name in cache:
#         return
#     relied_note_names: list[str] = list(cache[origin_note.name])
#     for relied_note_name in relied_note_names:
#         relied_note_link_types = cache[origin_note.name][relied_note_name]
#         if not relied_note_link_types:
#             continue
#         if not NoteLinkEnum.INFO_TO_INFO_VIA_NOTAT in relied_note_link_types:
#             continue
#         relied_note = VaultNote(origin_note.vault, name=relied_note_name)
#         notat_notes: list[VaultNote] = notation_notes_linked_in_see_also_section(
#             relied_note, origin_note.vault, as_vault_notes=True)
#         for notat_note in notat_notes:
#             _predict_one_direction_and_consolidate_cache(
#                 origin_note, notat_note, cache, predictor, format, note_data,
#                 skip_already_made_predictions, threshold)

In [ ]:
#| export machine_learning.note_linking
def predict_on_relied_notes_and_related_notat_notes(
        origin_note: VaultNote,
        relied_notes: VaultNote | list[VaultNote],
        cache: dict[str, dict[str, list[NoteLinkEnum]]],
        predictor: MultiLabelPipeline,
        format: Literal['bert', 't5'] = 'bert',
        note_data: Optional[dict[str, NoteData]] = None,
        skip_already_made_predictions: bool = True,
        predict_reverse_too: bool = False,
        threshold: float | dict[str, float] = 0.5,
        batch_size: int = 32
        ) -> None:
    """
    Predict linking between an origin note and multiple relied notes, 
    including secondary notation note checks, in optimized batches.
    """
    # Normalize input to a list
    relied_list = relied_notes if isinstance(relied_notes, list) else [relied_notes]
    if not relied_list:
        return

    # 1. Primary Prediction (Origin -> All Relied Notes)
    _predict_batch_and_consolidate_cache(
        origin_note, relied_list, cache, predictor, 
        format, note_data, batch_size, skip_already_made_predictions, threshold)

    # 2. Reverse Prediction (if requested)
    if predict_reverse_too:
        for r_note in relied_list:
            _predict_batch_and_consolidate_cache(
                r_note, [origin_note], cache, predictor, 
                format, note_data, batch_size, skip_already_made_predictions, threshold)

    # 3. Collect all notation notes that need checking based on primary results
    notation_notes_to_check = []
    if origin_note.name in cache:
        for r_note in relied_list:
            link_types = cache[origin_note.name].get(r_note.name, [])
            if NoteLinkEnum.INFO_TO_INFO_VIA_NOTAT in link_types:
                # Find notation notes mentioned in the 'See Also' of the info note
                found_notat = notation_notes_linked_in_see_also_section(
                    r_note, origin_note.vault, as_vault_notes=True)
                notation_notes_to_check.extend(found_notat)

    # 4. Final Batch Prediction for all collected notation notes
    if notation_notes_to_check:
        # Remove duplicates to avoid redundant predictions
        unique_notat = list({n.path: n for n in notation_notes_to_check}.values())
        _predict_batch_and_consolidate_cache(
            origin_note, unique_notat, cache, predictor, 
            format, note_data, batch_size, skip_already_made_predictions, threshold)

In [ ]:
#| hide
from unittest.mock import MagicMock, patch
from fastcore.test import *

def test_predict_on_relied_notes_and_related_notat_notes():
    # 1. Setup Mocks
    vault = "/mock/vault"
    origin_note = MagicMock(spec=VaultNote)
    origin_note.name = "Origin"
    origin_note.vault = vault
    
    relied_1 = MagicMock(spec=VaultNote)
    relied_1.name = "Relied1"
    relied_1.path = "path/to/relied1"
    
    notat_note = MagicMock(spec=VaultNote)
    notat_note.name = "Notat1"
    notat_note.path = "path/to/notat1"
    
    predictor = MagicMock()
    
    # 2. Patching orchestration and notation finding
    with patch('__main__._predict_batch_and_consolidate_cache') as mock_batch_call, \
         patch('__main__.notation_notes_linked_in_see_also_section') as mock_find_notat:
        
        # --- Test Case 1: Simple list of notes, no reverse, no notations ---
        cache = {} # Empty cache
        predict_on_relied_notes_and_related_notat_notes(
            origin_note, [relied_1], cache, predictor)
        
        # Should only be called once for the primary list
        test_eq(mock_batch_call.call_count, 1)
        mock_batch_call.assert_called_with(
            origin_note, [relied_1], cache, predictor, 
            'bert', None, 32, True, 0.5)

        mock_batch_call.reset_mock()

        # --- Test Case 2: Reverse Prediction requested ---
        predict_on_relied_notes_and_related_notat_notes(
            origin_note, [relied_1], cache, predictor, predict_reverse_too=True)
        
        # Called twice: once for Origin->Relied, once for Relied->Origin
        test_eq(mock_batch_call.call_count, 2)
        # Check the reverse call (second call)
        args = mock_batch_call.call_args_list[1][0]
        test_eq(args[0], relied_1)      # New origin
        test_eq(args[1], [origin_note]) # New relied list

        mock_batch_call.reset_mock()

        # --- Test Case 3: Secondary Notation Check ---
        # Simulate that the primary call found a 'VIA_NOTAT' relationship
        cache = {"Origin": {"Relied1": [NoteLinkEnum.INFO_TO_INFO_VIA_NOTAT]}}
        mock_find_notat.return_value = [notat_note]
        
        predict_on_relied_notes_and_related_notat_notes(
            origin_note, [relied_1], cache, predictor)
        
        # Should be called twice: 
        # 1. For [relied_1]
        # 2. For [notat_note] (because VIA_NOTAT was in cache)
        test_eq(mock_batch_call.call_count, 2)
        
        # Verify the second call was for the notation note
        last_call_args = mock_batch_call.call_args_list[1][0]
        test_eq(last_call_args[1], [notat_note])
        
        # --- Test Case 4: Deduplication of notations ---
        mock_batch_call.reset_mock()
        # Two info notes point to the SAME notation note
        relied_2 = MagicMock(spec=VaultNote)
        relied_2.name = "Relied2"
        relied_2.path = "path/to/relied2"
        
        cache["Origin"]["Relied2"] = [NoteLinkEnum.INFO_TO_INFO_VIA_NOTAT]
        mock_find_notat.return_value = [notat_note] # Both return the same notation note
        
        predict_on_relied_notes_and_related_notat_notes(
            origin_note, [relied_1, relied_2], cache, predictor)
        
        # Second call should still only have ONE notation note despite two info notes pointing to it
        notat_call_list = mock_batch_call.call_args_list[1][0][1]
        test_eq(len(notat_call_list), 1)

test_predict_on_relied_notes_and_related_notat_notes()

## Extract info notes that should give more information about a given info note

In [ ]:
#| export machine_learning.note_linking
from pathlib import Path
from typing import List, Dict, Any, Set

In [ ]:
#| export machine_learning.note_linking
def get_all_linked_info_notes(info_note: VaultNote, reference: str) -> List[VaultNote]:
    """
    Retrieves all info notes linked to the given info note.
    
    Aggregates dependencies from:
    1. The Link Cache (using `link_cache_note` and `parse_link_cache_note`).
    2. Dynamic content processing (using `links_from_text` and `ObsidianLink`).
    
    Args:
        info_note: The VaultNote to analyze.
        reference: The reference string (e.g., 'Algebra') used to locate the 
                   subvault and link cache.
        
    Returns:
        A list of unique VaultNote objects representing the dependencies.
    """
    # Use a dictionary keyed by note name to ensure uniqueness.
    # This handles cases where VaultNote equality is based on object identity.
    found_notes_map: Dict[str, VaultNote] = {}
    vault = info_note.vault
    
    # ---------------------------------------------------------
    # 1. Retrieve from Link Cache
    # ---------------------------------------------------------
    # Resolve subvault path using the index note anchor
    index_note = VaultNote(vault, name=f'_index_{reference}')
    
    if index_note.exists():
        subvault_path = index_note.path(relative=False).parent
        
        cache_note_obj = link_cache_note(subvault_path, reference, create_if_does_not_exist=False)
        
        if cache_note_obj.exists():
            link_types_cache = parse_link_cache_note(cache_note_obj)
            
            if info_note.name in link_types_cache:
                targets_dict = link_types_cache[info_note.name]
                
                valid_types = {
                    NoteLinkEnum.INFO_TO_INFO_IN_CONTENT, 
                    NoteLinkEnum.INFO_TO_INFO_VIA_NOTAT
                }
                
                for target_name, link_types_list in targets_dict.items():
                    if any(lt in valid_types for lt in link_types_list):
                        # Only create/add if not already found
                        if target_name not in found_notes_map:
                            target_note = VaultNote(vault, name=target_name)
                            if target_note.exists():
                                found_notes_map[target_name] = target_note
    else:
        # print(f"Debug: Index note '_index_{reference}' not found. Skipping cache lookup.")
        pass

    # ---------------------------------------------------------
    # 2. Retrieve from Processed Content (Dynamic Scan)
    # ---------------------------------------------------------
    mf = MarkdownFile.from_vault_note(info_note)
    
    processed_output = process_standard_information_note(
        mf, 
        vault, 
        remove_links=False, 
        remove_footnotes_to_embedded=False
    )
    
    processed_text = str(processed_output)
    
    links: List[ObsidianLink] = links_from_text(processed_text)
    
    for link in links:
        name = link.file_name
        
        # Optimization: Skip if we already have this note
        if name in found_notes_map:
            continue

        linked_note = VaultNote(vault, name=name)
        
        if not linked_note.exists():
            continue
            
        # Case A: Direct link to a Standard Info Note
        if note_is_of_type(linked_note, PersonalNoteTypeEnum.STANDARD_INFORMATION_NOTE):
            found_notes_map[name] = linked_note
            
        # Case B: Link to a Notation Note -> Get its Main Info Note
        elif note_is_of_type(linked_note, PersonalNoteTypeEnum.NOTATION_NOTE):
            main_info = main_of_notation(linked_note, as_note=True)
            if main_info and main_info.exists():
                # Use main_info.name as key to avoid duplicates
                found_notes_map[main_info.name] = main_info

    # Remove self-reference if present
    if info_note.name in found_notes_map:
        del found_notes_map[info_note.name]

    return list(found_notes_map.values())